In [25]:
%pip install SimpleITK
%pip install meshio
%pip install h5py


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 3.4 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
import SimpleITK as sitk
import dolfinx
from ufl import nabla_div, split, nabla_grad
import ufl
import math 
import meshio
import sys
import os
from mpi4py import MPI
from petsc4py.PETSc import ScalarType
from petsc4py import PETSc
from dolfinx.fem.petsc import LinearProblem
from matplotlib import pyplot as plt
from dolfinx.io import XDMFFile
from dolfinx.mesh import meshtags_from_entities
from dolfinx import log, default_scalar_type, fem, mesh, plot
from dolfinx.cpp.mesh import cell_entity_type
from dolfinx.io import distribute_entity_data
from dolfinx.graph import adjacencylist
from dolfinx.mesh import create_mesh
from dolfinx.cpp.mesh import to_type
from dolfinx.cpp.io import perm_gmsh
import numpy as np
import gmsh
import warnings
from IPython.display import clear_output
from dolfinx.fem.petsc import NonlinearProblem, assemble_matrix, assemble_vector, apply_lifting, create_vector, set_bc
from ufl import FacetNormal
import basix.ufl
from dolfinx import log, default_scalar_type, fem, mesh, plot
from dolfinx import cpp as _cpp
from dolfinx.fem.assemble import assemble_matrix
from dolfinx.fem import (Constant, Function, FunctionSpace, dirichletbc,
                         extract_function_spaces, form,
                         locate_dofs_geometrical, locate_dofs_topological, form, create_sparsity_pattern)
from dolfinx.fem.petsc import (NonlinearProblem, assemble_matrix, 
                               assemble_vector, apply_lifting, create_vector, 
                               set_bc, LinearProblem)
from dolfinx.io import XDMFFile
from dolfinx.mesh import (CellType, GhostMode, create_rectangle,
                          locate_entities_boundary)
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.nls.petsc import NewtonSolver
import dolfinx 

import scipy
import numpy as np
import ufl
from ufl import *
from mpi4py import MPI
from scipy.interpolate import LinearNDInterpolator, RegularGridInterpolator
from matplotlib import pyplot as plt
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter
from basix.ufl import element, mixed_element
from scipy.ndimage import distance_transform_edt

In [27]:
def TranslateSurface(i,Profile,Spacing):
    Vals = np.load(Profile+str(i)+'.npy')
    Displacements = np.zeros((Resolution,Resolution,Resolution,3))
    for I in range(Resolution):
        for J in range(Resolution):
            for K in range(Resolution):
                ThetaNow = int(Theta[I,J,K])
                Ur, Ut = Vals[ThetaNow,0], Vals[ThetaNow,1]
                Phi =  np.arctan((I-Resolution/2)/(J-Resolution/2+1e-5))
                Uz = np.cos(ThetaNow*np.pi/360)*Ur - np.sin(ThetaNow*np.pi/360)*Ut
                Ux = np.sin(ThetaNow*np.pi/360)*np.sin(Phi)*Ur*np.sign(J-Resolution/2+1e-6)
                Ux += np.sin(Phi)*np.cos(ThetaNow*np.pi/360)*Ut*np.sign(J-Resolution/2+1e-6)
                Uy = np.sin(ThetaNow*np.pi/360)*np.cos(Phi)*Ur*np.sign(J-Resolution/2) + np.cos(Phi)*np.sign(J-Resolution/2)*np.cos(ThetaNow*np.pi/360)*Ut
                Displacements[I,J,K,:] += np.array([Ux,Uy,-Uz])
    return Displacements    

In [28]:
def MakeMesh():
    warnings.filterwarnings("ignore")
    gmsh.initialize()
    gmsh.model.add("DFG 3D")

    Sphere = gmsh.model.occ.addSphere(Centre[0], Centre[1], Centre[2], RadiusObject)
    gmsh.model.occ.synchronize()
    
    volumes = gmsh.model.getEntities(dim=3)
    surfaces = gmsh.model.occ.getEntities(dim=2)
    
    Indent = []
    for surface in surfaces:
        Indent.append(surface[1])
    vol = [] 
    for volume in volumes:
        vol.append(volume[1])
    
    gmsh.model.addPhysicalGroup(2, Indent, 0)
    gmsh.model.setPhysicalName(2, 0, "Surface")
    gmsh.model.add_physical_group(3, vol, 1)
    gmsh.model.setPhysicalName(3, 1, "Volume")
    
    gmsh.model.mesh.field.add("Box", 1)
    gmsh.model.mesh.field.setNumber(1, "XMin", -RadiusObject)
    gmsh.model.mesh.field.setNumber(1, "XMax", RadiusObject)
    gmsh.model.mesh.field.setNumber(1, "ZMin", -RadiusObject)
    gmsh.model.mesh.field.setNumber(1, "ZMax", RadiusObject)
    gmsh.model.mesh.field.setNumber(1, "YMin", -RadiusObject)
    gmsh.model.mesh.field.setNumber(1, "YMax", RadiusObject)
    gmsh.model.mesh.field.setNumber(1, "VIn", MeshResolution)
    gmsh.model.mesh.field.setNumber(1, "VOut", MeshResolution*10)
    gmsh.model.mesh.field.setNumber(1, "Thickness", RadiusObject)
    
    gmsh.model.mesh.field.setAsBackgroundMesh(1)
    
    gmsh.model.occ.synchronize()
    gmsh.model.mesh.generate(3)
    
    mesh_name = Profile+"Sphere"
    gmsh.write(mesh_name+".msh")
    
    msh = meshio.read(f"{mesh_name}.msh")
    tetra_data = msh.cell_data_dict["gmsh:physical"]["tetra"]
    meshio.write(f"{mesh_name}.xdmf",meshio.Mesh(points=msh.points,cells={"tetra": msh.cells_dict["tetra"]},cell_data={"bnd_marker": [tetra_data]},),)
    tri_data = msh.cell_data_dict["gmsh:physical"]["triangle"]
    meshio.write(f"{mesh_name}_surf.xdmf",meshio.Mesh(points=msh.points,cells={"triangle": msh.cells_dict["triangle"]},cell_data={"bnd_marker": [tri_data]},),)

In [29]:
def SolveSystem(i,NLET,Indenter):
    mesh_name = Profile+"Sphere"

    with XDMFFile(MPI.COMM_WORLD, mesh_name+".xdmf", 'r') as xdmf_infile:
        Mesh = xdmf_infile.read_mesh(name='Grid')
        tags = xdmf_infile.read_meshtags(Mesh, name="Grid")
    Mesh.topology.create_connectivity(Mesh.topology.dim, Mesh.topology.dim-1)
    with XDMFFile(MPI.COMM_WORLD, mesh_name+"_surf.xdmf", 'r') as xdmf_infile:
        bound = xdmf_infile.read_meshtags(Mesh, name="Grid")  
    Mesh.topology.create_connectivity(Mesh.topology.dim-1, Mesh.topology.dim)

    VD = dolfinx.fem.functionspace(Mesh, ("CG", 1, (3,)))
    u_bc = fem.Function(VD)
    u_bc.interpolate(u_measured)
        
    dofs = fem.locate_dofs_topological(VD, bound.dim, bound.find(0))
    bcs = [fem.dirichletbc(u_bc, dofs)]
    v = ufl.TestFunction(VD)
    u = fem.Function(VD)

    d = len(u)
    I = ufl.variable(ufl.Identity(d))
    F = ufl.variable(I + ufl.grad(u))
    C = ufl.variable(F.T * F)
    Ic = ufl.variable(ufl.tr(C))
    J = ufl.variable(ufl.det(F))
    E = default_scalar_type(1.5e3)
    nu = default_scalar_type(0.4)
    mu = fem.Constant(Mesh, E / (2 * (1 + nu)))
    lmbda = fem.Constant(Mesh, E * nu / ((1 + nu) * (1 - 2 * nu)))
    if NLET:
        psi = (mu / 2) * (Ic - 3) - mu * ufl.ln(J) + (lmbda / 2) * (ufl.ln(J)) ** 2
        P = ufl.diff(psi, F)
    else:
        P = 2.0 * mu * ufl.sym(ufl.grad(u)) + lmbda * ufl.tr(ufl.sym(ufl.grad(u))) * I
    metadata = {"quadrature_degree": 4}
    ds = ufl.Measure("ds", domain=Mesh, subdomain_data=bound)
    dx = ufl.Measure("dx", domain=Mesh)
    residual = (ufl.inner(ufl.grad(v), P) * dx)
    problem = NonlinearProblem(residual,u,bcs=bcs,petsc_options_prefix="hyperelasticity")
    solver = problem.solver
    solver.setTolerances(rtol=1e-8, atol=1e-8, max_it=100)
    solver.getKSP().setType("preonly")
    solver.getKSP().getPC().setType("lu")
    solver.solve(None, problem.x)
    problem.u.x.scatter_forward()

    Normal = ufl.FacetNormal(Mesh)
    traction_expr = ufl.dot(P, Normal)

    ut = ufl.TrialFunction(VD)
    vt = ufl.TestFunction(VD)
    at = ufl.inner(ut, vt) * ds(0)
    Lt = ufl.inner(traction_expr, vt) * ds(0)
    problem = fem.petsc.LinearProblem(at,Lt,petsc_options={"ksp_type": "cg","pc_type": "jacobi"},petsc_options_prefix="traction_proj")
    traction_func = problem.solve()

    with dolfinx.io.XDMFFile(MPI.COMM_WORLD, mesh_name+str(i)+"_output_U_L"+str(int(NLET))+".xdmf", "w") as xdmf_outfile:
        xdmf_outfile.write_mesh(Mesh)
        u.name = "Displacement"
        xdmf_outfile.write_function(u)

    with dolfinx.io.XDMFFile(MPI.COMM_WORLD, mesh_name+str(i)+"_output_T_L"+str(int(NLET))+".xdmf", "w") as xdmf_outfile:
        xdmf_outfile.write_mesh(Mesh)
        traction_func.name = "Traction"
        xdmf_outfile.write_function(traction_func)
    
    Traction = fem.assemble_scalar(fem.form(ufl.sqrt(ufl.dot(traction_expr,traction_expr)) * ds(0)))
    return Traction

In [30]:
RadiusObject = 0.99
Centre = np.array([0,0,0])
MeshResolution = 0.05
Indenter = 'Surface'

Resolution = 200

Iterations = 7

StrainArrayHertz = np.array([0.01, 0.028, 0.082, 0.141, 0.214, 0.3, 0.398])
StrainArrayRing  = np.array([0.051, 0.0102, 0.153, 0.204, 0.306, 0.409, 0.511])
StrainArrayGauss = np.array([0.03, 0.066, 0.098, 0.131, 0.197, 0.262, 0.328])

In [31]:
Profile = 'Hertzian'

if Profile == 'Hertzian':
    StrainArray = StrainArrayHertz
elif Profile == 'Ring':
    StrainArray = StrainArrayRing
elif Profile == 'Gauss':
    StrainArray = StrainArrayGauss
else:
    print('Unknown profile')
    
X = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((Resolution,1,1)),Resolution,axis=1),Resolution,axis=2)
Y = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((1,Resolution,1)),Resolution,axis=0),Resolution,axis=2)
Z = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((1,1,Resolution)),Resolution,axis=0),Resolution,axis=1)
Theta = (np.pi-np.arctan2(np.sqrt(X**2+Y**2),Z))*360/np.pi
MeshScaleV = [(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1),(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1),(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1)]

IntegratedTraction = np.zeros((Iterations,2))

MakeMesh()

for i in range(Iterations):
    Displacements = TranslateSurface(i,Profile,Spacing)
    interp_V = RegularGridInterpolator(MeshScaleV, Displacements, bounds_error=False, fill_value=0.0,method='nearest')
    def u_measured(x):
        vals = interp_V(x.T)
        return vals.T
    IntegratedTraction[i,0] += SolveSystem(i,0,Indenter)
    IntegratedTraction[i,1] += SolveSystem(i,1,Indenter)
    
np.save(Profile+"IntegratedTraction.npy",IntegratedTraction)

Info    : Meshing 1D...
Info    : [ 40%] Meshing curve 2 (Circle)
Info    : Done meshing 1D (Wall 0.000264028s, CPU 0.000471s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Sphere, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.366455s, CPU 0.363838s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 6070 nodes...
Info    : Done tetrahedrizing 6078 nodes (Wall 0.0627968s, CPU 0.055022s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.169146s, CPU 0.157672s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 22.1945 (nodes removed 0 0)
Info    : It. 500 - 500 nodes created - worst tet radius 3.19131 (nodes removed 0 0)
Info    : It. 1000 - 1000 nodes created - worst tet radius 2.60031 (nodes removed 0 0)
Info    : It. 1500 - 1500 nodes created - worst tet radi

NameError: name 'Spacing' is not defined

In [ ]:
plt.plot(StrainArray,IntegratedTraction,label=['LET','Neo-Hookean'])
plt.title('Traction-Indentation curves for LE and NH with ' +Profile +' indenter')
plt.xlabel('Maximum relative indentation')
plt.ylabel('Total traction [Pa]')
plt.legend()
plt.savefig("TractionStrain"+Profile+".png")
plt.show()

In [ ]:
Profile = 'Ring'

if Profile == 'Hertzian':
    StrainArray = StrainArrayHertz
elif Profile == 'Ring':
    StrainArray = StrainArrayRing
elif Profile == 'Gauss':
    StrainArray = StrainArrayGauss
else:
    print('Unknown profile')
    
X = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((Resolution,1,1)),Resolution,axis=1),Resolution,axis=2)
Y = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((1,Resolution,1)),Resolution,axis=0),Resolution,axis=2)
Z = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((1,1,Resolution)),Resolution,axis=0),Resolution,axis=1)
Theta = (np.pi-np.arctan2(np.sqrt(X**2+Y**2),Z))*360/np.pi
MeshScaleV = [(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1),(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1),(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1)]

IntegratedTraction = np.zeros((Iterations,2))

MakeMesh()

for i in range(Iterations):
    Displacements = TranslateSurface(i,Profile,Spacing)
    interp_V = RegularGridInterpolator(MeshScaleV, Displacements, bounds_error=False, fill_value=0.0,method='nearest')
    def u_measured(x):
        vals = interp_V(x.T)
        return vals.T
    IntegratedTraction[i,0] += SolveSystem(i,0,Indenter)
    IntegratedTraction[i,1] += SolveSystem(i,1,Indenter)
    
np.save(Profile+"IntegratedTraction.npy",IntegratedTraction)

In [ ]:
plt.plot(StrainArray,IntegratedTraction,label=['LET','Neo-Hookean'])
plt.title('Traction-Indentation curves for LE and NH with ' +Profile +' indenter')
plt.xlabel('Maximum relative indentation')
plt.ylabel('Total traction [Pa]')
plt.legend()
plt.savefig("TractionStrain"+Profile+".png")
plt.show()

In [32]:
Profile = 'Gaussian'

if Profile == 'Hertzian':
    StrainArray = StrainArrayHertz
elif Profile == 'Ring':
    StrainArray = StrainArrayRing
elif Profile == 'Gauss':
    StrainArray = StrainArrayGauss
else:
    print('Unknown profile')
    
X = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((Resolution,1,1)),Resolution,axis=1),Resolution,axis=2)
Y = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((1,Resolution,1)),Resolution,axis=0),Resolution,axis=2)
Z = np.repeat(np.repeat((2*np.arange(Resolution)/Resolution-1).reshape((1,1,Resolution)),Resolution,axis=0),Resolution,axis=1)
Theta = (np.pi-np.arctan2(np.sqrt(X**2+Y**2),Z))*360/np.pi
MeshScaleV = [(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1),(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1),(2*(Resolution/(Resolution-1))*np.arange(Resolution)/Resolution-1)]

IntegratedTraction = np.zeros((Iterations,2))

MakeMesh()

for i in range(Iterations):
    Displacements = TranslateSurface(i,Profile,Spacing)
    interp_V = RegularGridInterpolator(MeshScaleV, Displacements, bounds_error=False, fill_value=0.0,method='nearest')
    def u_measured(x):
        vals = interp_V(x.T)
        return vals.T
    IntegratedTraction[i,0] += SolveSystem(i,0,Indenter)
    IntegratedTraction[i,1] += SolveSystem(i,1,Indenter)
    
np.save(Profile+"IntegratedTraction.npy",IntegratedTraction)

Unknown profile


Info    : Meshing 1D...
Info    : [ 40%] Meshing curve 2 (Circle)
Info    : Done meshing 1D (Wall 0.000131809s, CPU 0.000218s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Sphere, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.350217s, CPU 0.348112s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 6070 nodes...
Info    : Done tetrahedrizing 6078 nodes (Wall 0.0594726s, CPU 0.055444s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.151711s, CPU 0.148241s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 22.1945 (nodes removed 0 0)
Info    : It. 500 - 500 nodes created - worst tet radius 3.19131 (nodes removed 0 0)
Info    : It. 1000 - 1000 nodes created - worst tet radius 2.60031 (nodes removed 0 0)
Info    : It. 1500 - 1500 nodes created - worst tet radi

NameError: name 'Spacing' is not defined

In [ ]:
plt.plot(StrainArray,IntegratedTraction,label=['LET','Neo-Hookean'])
plt.title('Traction-Indentation curves for LE and NH with ' +Profile +' indenter')
plt.xlabel('Maximum relative indentation')
plt.ylabel('Total traction [Pa]')
plt.legend()
plt.savefig("TractionStrain"+Profile+".png")
plt.show()